# Ingesta de países de producción

Lee la carpeta `production_country` (JSON multilínea) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/production_countries` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee la carpeta `production_country` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
production_country_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/production_country")
    )
display(production_country_df)

In [0]:
production_country_shema = StructType([
    StructField("movieId", IntegerType()),
    StructField("countryId", IntegerType())
])

production_country_df = (
    spark.read
        .schema(production_country_shema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/production_country")
)
display(production_country_df)

### Transformación de columnas

Renombra `movieId` y `countryId` a snake_case y agrega `env`.

In [0]:
production_country_df = (
    production_country_df
        .withColumnsRenamed(
            {
                "movieId": "movie_Id",
                "countryId": "country_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env)
            }
        )
)

production_country_df = add_ingestion_date(production_country_df)

display(production_country_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/production_countries` en modo `overwrite`.

In [0]:
(
    production_country_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/production_countries")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")